# Tutorial: In-Silico Gene Knockout Perturbation for a Single Section (Terra model)


## 1. Imports and configuration

Same imports/paths as the original script. The original script takes `--output_dir` and
`--device` as CLI args — here we just set them as plain variables so you can edit them
directly.


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import scanpy as sc


# GeomLoss for Sinkhorn (W1/W2), Energy, MMD(Gaussian)
from geomloss import SamplesLoss
import torch
# <-- EDIT THESE (equivalent to the original --output_dir / --device CLI args)
OUTPUT_DIR = "/lustre/scratch126/cellgen/lotfollahi/am74/terra_notebooks/kidney_analysis/Healthy_section/reproducibility/Results_ICB_new"
DEVICE     = "cuda"   # or "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
# ── Shared paths ─────────────────────────────────────────────────────────────
ADATA_PATH = (
    "/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/"
    "NEMO_UpdatedTokenisationandEmbedding_November22nd25/"
    "Labeltransfer/forshare/"
    "nemokidneyxeniumatlas_annotated_FINALFORFEBRUARY.h5ad"
)
DATASET_DIR = (
    "/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/"
    "NEMO_UpdatedTokenisationandEmbedding_November22nd25/"
    "tokenised_xenium_common_qc/"
)
MODEL_FOLDER_PATH = (
    "/nfs/team361/sb75/terra-reproducibility/"
    "artifacts/models/01112025_081731_614"
)

sys.path.append("/nfs/team361/am74/nemo/nichejepa-reproducibility/utils")

import torch
from datasets import load_from_disk, concatenate_datasets, Dataset
from geomloss import SamplesLoss

from terra.inference import (
    embed_dataset,
    harmonize_adata,
    perturb_dataset,
)
from terra.preprocessors import construct_neighbor_graph
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (no display needed)
import matplotlib.pyplot as plt

# Improve UMAP/Leiden reproducibility
os.environ["NUMBA_CPU_NAME"] = "generic"

print(f"PyTorch CUDA version : {torch.version.cuda}")
print(f"CUDA available       : {torch.cuda.is_available()}")


/nfs/team361/mv10/terra_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/team361/mv10/terra_venv/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/team361/mv10/terra_venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


PyTorch CUDA version : 12.4
CUDA available       : True


## 2. Load the full atlas and filter to "Healthy" samples

We load the full annotated Xenium kidney atlas, reset `X` to raw counts, derive a
`Diagnosis` column from `RCC_Treatment`, and keep only the `Healthy` cells (this is the
same cohort the array job iterates over, one slice at a time).


In [3]:
print("Loading AnnData …")
adata_full = sc.read(ADATA_PATH)
adata_full.X = adata_full.layers["counts"].copy()

def assign_diagnosis(treatment):
    if treatment == "na":
        return "Healthy"
    elif treatment in ["Nivo", "Ipi+Nivo"]:
        return "ICB exposure"
    elif treatment in ["Cabo+Nivo", "Ipi+Nivo+Sutinib"]:
        return "ICB + TKI exposure"
    else:
        return pd.NA

adata_full.obs["Diagnosis"] = adata_full.obs["RCC_Treatment"].map(assign_diagnosis)
adata_full.obs["Diagnosis"] = adata_full.obs["Diagnosis"].astype("category")
adata_full = adata_full[adata_full.obs["Diagnosis"] == "Healthy"].copy()

print(f"Healthy cells: {adata_full.n_obs:,}")


Loading AnnData …
Healthy cells: 1,080,937


## 3. Pick one slice (section) to process

In the original script, `section_index` (a CLI arg, one per LSF array task) selects a
slice from the sorted list of `NEMO_K_SLICE_ID`s. Here we hardcode `SECTION_INDEX`
directly — edit it (or set `SLICE_ID` directly) to choose a different slice.


In [4]:
slice_ids = sorted(adata_full.obs["NEMO_K_SLICE_ID"].unique().tolist())
print(f"Found {len(slice_ids)} Healthy slices: {slice_ids}")

# <-- EDIT THIS: which slice to process (0-based index into slice_ids above)
SECTION_INDEX = 6

if SECTION_INDEX >= len(slice_ids):
    raise ValueError(
        f"SECTION_INDEX {SECTION_INDEX} is out of range "
        f"(only {len(slice_ids)} slices available)."
    )

slice_id = slice_ids[SECTION_INDEX]
print(f"\nProcessing slice [{SECTION_INDEX}] -> {slice_id}")


Found 9 Healthy slices: ['K001', 'K002', 'K008', 'K009', 'K010', 'K012', 'K014', 'K015', 'K016']

Processing slice [6] -> K014


In [ ]:
out_csv = os.path.join(OUTPUT_DIR, f"{slice_id}_insilicoW2.csv")
if os.path.exists(out_csv):
    print(f"NOTE: {out_csv} already exists. Re-running will overwrite it if you save at the end.")


## 4. Build the spatial neighbor graph and neighborhood expression layers



In [5]:
print("Building neighbor graph …")
adata_full = construct_neighbor_graph(
    adata_full,
    n_neighs=10,
    radius=None,
    delaunay=False,
    include_self_loop=False,
    compute_neighbor_counts=False,
    batch_key="NEMO_K_SLICE_ID",
)

adata_full.layers["counts_neighborhood"] = (
    adata_full.obsp["spatial_connectivities"].T @ adata_full.layers["counts"]
)
adata_full.layers["counts_neighborhood"] += adata_full.layers["counts"]

adata_full.X = adata_full.layers["counts_neighborhood"].copy()
sc.pp.normalize_total(adata_full, target_sum=1e4)
sc.pp.log1p(adata_full)
adata_full.layers["lognorm_neighborhood"] = adata_full.X.copy()
adata_full.X = adata_full.layers["lognorm"].copy()
adata_full.X = adata_full.layers["counts"].copy()

print("Neighbor graph and neighborhood layers built.")


Building neighbor graph …


/nfs/team361/mv10/terra_pr_wt/src/terra/preprocessors/graph.py:114: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_idx = adata.obs.reset_index().groupby(batch_key).head(1).index
INFO:terra.preprocessors.graph:
Processing batch K002...
INFO:terra.preprocessors.graph:Loading data...
INFO:terra.preprocessors.graph:Computing spatial neighborhood graph...
/nfs/team361/mv10/terra_venv/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
INFO:terra.preprocessors.graph:
Processing batch K008...
INFO:terra.preprocessors.graph:Loading data...
INFO:terra.preprocessors.graph:Computing spatial neighborhood graph...
/nfs/team361/mv10/terra_venv/lib/python3.10/site-package

Neighbor graph and neighborhood layers built.


## 5. Subset the AnnData to the chosen slice

In [6]:
adata = adata_full[adata_full.obs["NEMO_K_SLICE_ID"] == slice_id].copy()
print(f"adata cells for {slice_id}: {adata.n_obs}")
del adata_full   # free memory


adata cells for K014: 123563


## 6. Load the tokenised HuggingFace dataset and match it to this slice


In [7]:
print("Loading tokenised datasets …")
subdirs = [
    os.path.join(DATASET_DIR, d)
    for d in sorted(os.listdir(DATASET_DIR))
    if os.path.isdir(os.path.join(DATASET_DIR, d))
]
print(f"  Found {len(subdirs)} dataset folders")
datasets_list = [load_from_disk(p) for p in subdirs]
full_dataset = concatenate_datasets(datasets_list)
print(f"  Concatenated dataset: {full_dataset}")

Loading tokenised datasets …
  Found 15 dataset folders
  Concatenated dataset: Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 1538328
})


In [8]:
cell_id_set = set(adata.obs["cell_id"])
cell_id_series = full_dataset.select_columns(["cell_id"]).to_pandas()["cell_id"]
mask = cell_id_series.isin(cell_id_set)
indices = mask[mask].index.tolist()

df_shared = full_dataset.select(indices).to_pandas()
hf_subset = Dataset.from_pandas(df_shared)
torch_subset = hf_subset.with_format("torch", columns=[
    "rel_x_coord", "rel_y_coord", "cell_id",
    "gene_tokens", "gene_expr", "n_nonzero_tokens",
])
print(f"Torch dataset ready: {len(torch_subset):,} cells")
del full_dataset, datasets_list   # free memory


Torch dataset ready: 123,605 cells


In [9]:
# Deduplicate adata by cell_id
adata.obs["cell_id"] = adata.obs_names
adata = adata[~adata.obs["cell_id"].duplicated(keep="first")].copy()
print(f"adata after dedup: {adata.n_obs} cells")


adata after dedup: 123563 cells


In [10]:
# Deduplicate torch_subset by cell_id
seen = set()
keep_indices = []
for i, cid in enumerate(torch_subset["cell_id"]):
    if cid not in seen:
        seen.add(cid)
        keep_indices.append(i)
torch_subset = torch_subset.select(keep_indices)
print(f"torch_subset after dedup: {len(torch_subset)} cells")


torch_subset after dedup: 123563 cells


## 7. Harmonize the AnnData

In [11]:
adata = harmonize_adata(adata)
adata


INFO:terra.inference.harmonize:STEP 1: DATA VALIDATION...
INFO:terra.inference.harmonize:Checking that adata.X contains raw counts...
INFO:terra.inference.harmonize:✓ adata.X contains raw counts (integer values).
INFO:terra.inference.harmonize:STEP 2: ADDING ENSEMBL IDS...
INFO:terra.inference.harmonize:Adding ensembl IDs from release 111...
INFO:terra.inference.harmonize:Make sure this ensembl release is aligned with pretraining.
INFO:terra.inference.harmonize:Current ensembl release used for pretraining is 111.
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/team361/mv10/.home_redirect/cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/team361/mv10/.home_redirect/cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/team361/mv10/.home_redirect/cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRC

AnnData object with n_obs × n_vars = 123563 × 4873
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_K_PATIENT_ID', 'Corpus_ID', 'Study_patient_id', 'Data_source', 'Platform', 'Tissue_subregion', 'Status', 'Diagnosis', 'Type', 'Age', 'Age_grouping', 'Sex', 'eGFR', 'eGFR_group', 'BMI', 'Obesity_status', 'eGFR baseline', 'eGFR change', 'RCC_Treatment', 'batch', 'NEMO_cell_label', 'NEMO_neighborhood_label', 'tech', 'scANVI_pred', 'scANVI_confidence', 'slice', 'NEMO_neighborhood_annotation', 'leiden', 'highlight_neigh', 'celltype_simplified', 'celltype_broadest', 'NEMO_updated_niche', 'score_r9_cluster6_0_Angiogenesis_GP', 'score_r9_cluster5_2_T_cell_migration_and_activation_GP', 'score_r9_cluster5_1_Neutrophil_activation_and_degranulation_GP', 'score_r9_cluster5_0_Cytokine_signalling_GP', 'score_r9_cluster4_4_Endothelial_pericyte_vasoactive_GP', 'score_r9_cluster4_3_Stromal_remodelling_GP', 'score_r9_cluster4_2_Glomerular_vascular_interface_GP', 'score_r9_cluster4_1_Perivascu

## 8. Define the in-silico knockout perturbation

We build a `perturb_df` describing which genes to knock out, on which cells, and at
which "target" (the cell's own expression, its neighborhood, or both). Here every gene
in `genes` is knocked out both in the cell itself and in its neighborhood, for **all**
cells (`perturbed_cell_id = "all"`).


In [12]:
genes = [
"ENSG00000168743","ENSG00000184937","ENSG00000151276"]
perturb_df = pd.DataFrame({
    "perturbed_cell_id": "all",
    "perturbed_ensembl_id": np.repeat(genes, 2),
    "perturbation_target": ["cell", "neighborhood"] * len(genes),
    "perturbation_type": "knockout",
    "foldchange": np.nan,
})
perturb_df


,perturbed_cell_id,perturbed_ensembl_id,perturbation_target,perturbation_type,foldchange
0,all,ENSG00000168743,cell,knockout,NaN
1,all,ENSG00000168743,neighborhood,knockout,NaN
2,all,ENSG00000184937,cell,knockout,NaN
3,all,ENSG00000184937,neighborhood,knockout,NaN
4,all,ENSG00000151276,cell,knockout,NaN
5,all,ENSG00000151276,neighborhood,knockout,NaN


## 9. Run the perturbation and re-embed the perturbed cells

`perturb_dataset` applies the knockout(s) to the tokenised dataset and returns only the
cells actually affected (`return_only_perturbed_cells=True`). We then print how many
cells were impacted by each individual perturbation row, and finally run
`embed_dataset` on the perturbed cells to get their new (post-knockout) embeddings.


In [13]:
print("Running perturbation …")
perturbed_dataset = perturb_dataset(
    dataset=torch_subset,
    perturb_df=perturb_df,
    nproc=1,
    model_folder_path=MODEL_FOLDER_PATH,
    return_only_perturbed_cells=True,
    pad_gene_tokens=True,
    adjust_positions=True,
    return_perturbation_flags=True,
)

for idx, row in perturb_df.iterrows():
    count = sum(f.item() for f in perturbed_dataset[f"pert_flag_idx_{idx}"])
    print(f"  {count}/{len(torch_subset)} cells impacted by perturbation {idx}.")


INFO:terra.inference.perturb:Applying perturbations using dataframe:
  perturbed_cell_id perturbed_ensembl_id perturbation_target  \
0               all      ENSG00000168743                cell   
1               all      ENSG00000168743        neighborhood   
2               all      ENSG00000184937                cell   
3               all      ENSG00000184937        neighborhood   
4               all      ENSG00000151276                cell   
5               all      ENSG00000151276        neighborhood   

  perturbation_type  foldchange perturbed_gene_token  
0          knockout         NaN                12305  
1          knockout         NaN                12305  
2          knockout         NaN                15266  
3          knockout         NaN                15266  
4          knockout         NaN                 9350  
5          knockout         NaN                 9350  


Running perturbation …


Map (num_proc=1): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 123563/123563 [05:54<00:00, 348.23 examples/s]


  5093/123563 cells impacted by perturbation 0.
  28502/123563 cells impacted by perturbation 1.
  244/123563 cells impacted by perturbation 2.
  2047/123563 cells impacted by perturbation 3.
  1925/123563 cells impacted by perturbation 4.
  16432/123563 cells impacted by perturbation 5.


In [17]:
print("Embedding perturbed dataset …")
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=MODEL_FOLDER_PATH,
    emb_layer=None,
    agg_excluded_genes=None,
    top_k=None,
    batch_size=128,
    pin_memory=False,
    num_workers=12,
    include_spatial_cell_emb=True,
    return_token_embeddings=False,
    ignore_spc_tokens=True,
)


INFO:terra.inference.embed:STEP 1: LOADING CONFIG...
INFO:terra.inference.embed:STEP 2: GENERATING EMBEDDINGS...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.


Embedding perturbed dataset …


INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
    (pos_embed): Embedding(2

## 10. Store the perturbed embeddings back into the AnnData


In [18]:
adata.obs_names = adata.obs["cell_id"].str.extract(r"_([a-z]+-\d+)-slice")[0]

for emb_key in ["neighborhood_emb", "cell_emb"]:
    new_emb = adata.obsm[emb_key].copy()
    perturbed_cell_ids = list(perturbed_dataset["cell_id"])
    mask_emb = adata.obs_names.isin(perturbed_cell_ids)
    perturbed_index = {cid: i for i, cid in enumerate(perturbed_cell_ids)}
    new_emb[mask_emb] = perturbed_output_embed[emb_key][
        [perturbed_index[cid] for cid in adata.obs_names[mask_emb]]
    ]
    adata.obsm[f"{emb_key}_perturb_case3"] = new_emb

    obs_names_to_pos = {cid: i for i, cid in enumerate(adata.obs_names)}
    valid_perturbed = [cid for cid in perturbed_cell_ids if cid in obs_names_to_pos]
    for cid in valid_perturbed[:100]:
        ai = obs_names_to_pos[cid]
        pi = perturbed_index[cid]
        np.testing.assert_allclose(
            adata.obsm[f"{emb_key}_perturb_case3"][ai],
            perturbed_output_embed[emb_key][pi],
        )

print("Perturbed embeddings stored as '<emb>_perturb_case3' in adata.obsm.")


Perturbed embeddings stored as '<emb>_perturb_case3' in adata.obsm.


## 11. Post perturbation analysis


In [19]:



def _l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-normalize each row (sample) of X."""
    X = np.asarray(X, dtype=np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + eps)


def _agg_fn_from_name(agg):
    if isinstance(agg, str):
        agg_name = agg.lower()
        if agg_name == "min":
            return np.min
        if agg_name == "mean":
            return np.mean
        if agg_name == "median":
            return np.median
        raise ValueError("agg must be 'min', 'mean', 'median', or a callable")
    if callable(agg):
        return agg
    raise ValueError("agg must be 'min', 'mean', 'median', or a callable")


def _geomloss_distance_pointcloud(
    X: np.ndarray,
    Y: np.ndarray,
    loss: str,
    p: int | None = None,
    blur: float = 0.01,
    backend: str = "tensorized",
    device: str | None = None,
) -> float:
    """
    Compute a *distribution* distance between two point clouds X and Y using GeomLoss.
    X, Y: arrays of shape (n_points, n_dims). Interpreted as samples from distributions.
    Returns a scalar.
    """
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"X and Y must be 2D arrays. Got {X.shape}, {Y.shape}.")
    if X.shape[1] != Y.shape[1]:
        raise ValueError(f"X and Y must have same dimensionality. Got {X.shape[1]} vs {Y.shape[1]}.")

    # Convert to torch tensors
    tX = torch.from_numpy(X)
    tY = torch.from_numpy(Y)

    if device is not None:
        tX = tX.to(device)
        tY = tY.to(device)

    if loss == "sinkhorn":
        if p not in (1, 2):
            raise ValueError("For loss='sinkhorn', p must be 1 or 2.")
        loss_fn = SamplesLoss(loss="sinkhorn", p=p, blur=blur, backend=backend)
    elif loss == "energy":
        loss_fn = SamplesLoss(loss="energy", blur=blur, backend=backend)
    elif loss == "gaussian":
        loss_fn = SamplesLoss(loss="gaussian", blur=blur, backend=backend)
    else:
        raise ValueError("loss must be one of {'sinkhorn','energy','gaussian'}.")

    val = loss_fn(tX, tY)
    return float(val.detach().cpu().item())


def summarize_w1_by_label(
    adata,
    label_key,
    labels=None,
    pairs=None,
    agg="mean",
    sort_by=None,
    ascending=True,
    eps=1e-12,
    ignore_zeros=False,
    blur=0.01,
    backend="tensorized",
    device=None,
    threshold=None,
):
    """
    Compute W1 (Sinkhorn, p=1) between *point-cloud distributions* (rows are points),
    per label in adata.obs[label_key], for each (A_key,B_key) pair in adata.obsm.

    L2-normalizes rows of both embeddings before computing.
    Returns a DataFrame: ["label","n_cells", out_col...], optionally sorted.
    """
    if pairs is None:
        pairs = [
            ("cell_emb", "cell_emb_perturb_case1", "cell_emb_w1"),
            ("spatial_cell_emb", "spatial_cell_emb_perturb_case1", "spatial_cell_emb_w1"),
            ("neighborhood_emb", "neighborhood_emb_perturb_case1", "neighborhood_emb_w1"),
        ]
    if labels is None:
        labels = sorted(pd.unique(adata.obs[label_key]))

    agg_fn = _agg_fn_from_name(agg)

    rows = []
    for label in labels:
        adata_sub = adata[adata.obs[label_key] == label]
        row = {"label": label, "n_cells": int(adata_sub.n_obs)}

        for A_key, B_key, out_col in pairs:
            XA = _l2_normalize_rows(np.asarray(adata_sub.obsm[A_key]), eps=eps)
            XB = _l2_normalize_rows(np.asarray(adata_sub.obsm[B_key]), eps=eps)

            if threshold is not None and XA.shape[0] > threshold:
                idx = np.random.choice(XA.shape[0], size=int(threshold), replace=False)
                XA = XA[idx]
                XB = XB[idx]


            d = _geomloss_distance_pointcloud(
                XA, XB, loss="sinkhorn", p=1, blur=blur, backend=backend, device=device
            )

            vals = np.array([d], dtype=float)
            if ignore_zeros:
                vals = vals[vals != 0]

            row[out_col] = float("nan") if vals.size == 0 else float(agg_fn(vals))
        rows.append(row)

    df = pd.DataFrame(rows)
    if sort_by is not None:
        df = df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    return df


def summarize_w2_by_label(
    adata,
    label_key,
    labels=None,
    pairs=None,
    agg="mean",
    sort_by=None,
    ascending=True,
    eps=1e-12,
    ignore_zeros=False,
    blur=0.01,
    backend="tensorized",
    device=None,
    threshold=None,
):
    """
    Compute W2 (Sinkhorn, p=2) between point-cloud distributions per label.
    L2-normalizes rows before computing.
    """
    if pairs is None:
        pairs = [
            ("cell_emb", "cell_emb_perturb_case1", "cell_emb_w2"),
            ("spatial_cell_emb", "spatial_cell_emb_perturb_case1", "spatial_cell_emb_w2"),
            ("neighborhood_emb", "neighborhood_emb_perturb_case1", "neighborhood_emb_w2"),
        ]
    if labels is None:
        labels = sorted(pd.unique(adata.obs[label_key]))

    agg_fn = _agg_fn_from_name(agg)

    rows = []
    for label in labels:
        adata_sub = adata[adata.obs[label_key] == label]
        row = {"label": label, "n_cells": int(adata_sub.n_obs)}

        for A_key, B_key, out_col in pairs:
            XA = _l2_normalize_rows(np.asarray(adata_sub.obsm[A_key]), eps=eps)
            XB = _l2_normalize_rows(np.asarray(adata_sub.obsm[B_key]), eps=eps)

            if threshold is not None and XA.shape[0] > threshold:
                idx = np.random.choice(XA.shape[0], size=int(threshold), replace=False)
                XA = XA[idx]
                XB = XB[idx]

            d = _geomloss_distance_pointcloud(
                XA, XB, loss="sinkhorn", p=2, blur=blur, backend=backend, device=device
            )

            vals = np.array([d], dtype=float)
            if ignore_zeros:
                vals = vals[vals != 0]

            row[out_col] = float("nan") if vals.size == 0 else float(agg_fn(vals))
        rows.append(row)

    df = pd.DataFrame(rows)
    if sort_by is not None:
        df = df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    return df


def summarize_energy_by_label(
    adata,
    label_key,
    labels=None,
    pairs=None,
    agg="mean",
    sort_by=None,
    ascending=True,
    eps=1e-12,
    ignore_zeros=False,
    blur=0.5,
    backend="tensorized",
    device=None,
    threshold=None,
):
    """
    Compute Energy distance (GeomLoss 'energy') between point-cloud distributions per label.
    L2-normalizes rows before computing.
    """
    if pairs is None:
        pairs = [
            ("cell_emb", "cell_emb_perturb_case1", "cell_emb_energy"),
            ("spatial_cell_emb", "spatial_cell_emb_perturb_case1", "spatial_cell_emb_energy"),
            ("neighborhood_emb", "neighborhood_emb_perturb_case1", "neighborhood_emb_energy"),
        ]
    if labels is None:
        labels = sorted(pd.unique(adata.obs[label_key]))

    agg_fn = _agg_fn_from_name(agg)

    rows = []
    for label in labels:
        adata_sub = adata[adata.obs[label_key] == label]
        row = {"label": label, "n_cells": int(adata_sub.n_obs)}

        for A_key, B_key, out_col in pairs:
            XA = _l2_normalize_rows(np.asarray(adata_sub.obsm[A_key]), eps=eps)
            XB = _l2_normalize_rows(np.asarray(adata_sub.obsm[B_key]), eps=eps)

            if threshold is not None and XA.shape[0] > threshold:
                idx = np.random.choice(XA.shape[0], size=int(threshold), replace=False)
                XA = XA[idx]
                XB = XB[idx]

            d = _geomloss_distance_pointcloud(
                XA, XB, loss="energy", p=None, blur=blur, backend=backend, device=device
            )

            vals = np.array([d], dtype=float)
            if ignore_zeros:
                vals = vals[vals != 0]

            row[out_col] = float("nan") if vals.size == 0 else float(agg_fn(vals))
        rows.append(row)

    df = pd.DataFrame(rows)
    if sort_by is not None:
        df = df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    return df


def summarize_mmd_by_label(
    adata,
    label_key,
    labels=None,
    pairs=None,
    agg="mean",
    sort_by=None,
    ascending=True,
    eps=1e-12,
    ignore_zeros=False,
    blur=0.5,
    backend="tensorized",
    device=None,
    threshold=None,
):
    """
    Compute Gaussian MMD-like loss (GeomLoss 'gaussian') between point-cloud distributions per label.
    L2-normalizes rows before computing.
    """
    if pairs is None:
        pairs = [
            ("cell_emb", "cell_emb_perturb_case1", "cell_emb_mmd"),
            ("spatial_cell_emb", "spatial_cell_emb_perturb_case1", "spatial_cell_emb_mmd"),
            ("neighborhood_emb", "neighborhood_emb_perturb_case1", "neighborhood_emb_mmd"),
        ]
    if labels is None:
        labels = sorted(pd.unique(adata.obs[label_key]))

    agg_fn = _agg_fn_from_name(agg)

    rows = []
    for label in labels:
        adata_sub = adata[adata.obs[label_key] == label]
        row = {"label": label, "n_cells": int(adata_sub.n_obs)}

        for A_key, B_key, out_col in pairs:
            XA = _l2_normalize_rows(np.asarray(adata_sub.obsm[A_key]), eps=eps)
            XB = _l2_normalize_rows(np.asarray(adata_sub.obsm[B_key]), eps=eps)

            if threshold is not None and XA.shape[0] > threshold:
                idx = np.random.choice(XA.shape[0], size=int(threshold), replace=False)
                XA = XA[idx]
                XB = XB[idx]

            d = _geomloss_distance_pointcloud(
                XA, XB, loss="gaussian", p=None, blur=blur, backend=backend, device=device
            )

            vals = np.array([d], dtype=float)
            if ignore_zeros:
                vals = vals[vals != 0]

            row[out_col] = float("nan") if vals.size == 0 else float(agg_fn(vals))
        rows.append(row)

    df = pd.DataFrame(rows)
    if sort_by is not None:
        df = df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    return df

In [20]:
pert_genes = ['NPNT', 'WT1', 'MAGI1']
for pert_idx in range(len(pert_genes) * 2):
    ind_pert_cell_ids = list(
        perturbed_dataset.filter(lambda x: x[f"pert_flag_idx_{pert_idx}"])["cell_id"]
    )
    print(f"  pert_idx={pert_idx}: {len(ind_pert_cell_ids)} affected cells")


Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42930/42930 [00:03<00:00, 12958.66 examples/s]


  pert_idx=0: 5093 affected cells


Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42930/42930 [00:02<00:00, 14829.53 examples/s]


  pert_idx=1: 28502 affected cells


Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42930/42930 [00:02<00:00, 14757.29 examples/s]


  pert_idx=2: 244 affected cells


... (2 outputs left out)

Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42930/42930 [00:02<00:00, 15424.96 examples/s]


  pert_idx=4: 1925 affected cells


Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42930/42930 [00:02<00:00, 15491.25 examples/s]


  pert_idx=5: 16432 affected cells


In [21]:
affected_cell_ids = set(perturbed_dataset["cell_id"])
adata_sub = adata[adata.obs_names.isin(affected_cell_ids)].copy()

In [22]:
print("Computing W2 …")
df_w2 = summarize_w2_by_label(
    adata_sub,
    pairs=[("cell_emb",        "cell_emb_perturb_case3",        "cell_emb"),
           ("neighborhood_emb", "neighborhood_emb_perturb_case3", "neighborhood_emb")],
    label_key="NEMO_updated_niche",
    agg="mean",
    sort_by="neighborhood_emb",
    ascending=False,
    device=DEVICE,
    threshold=None,
)
df_w2 = df_w2[df_w2["n_cells"] > 50]
df_w2


Computing W2 …


,label,n_cells,cell_emb,neighborhood_emb
0,Glomerular niche,9636,0.001916,0.001903
1,Perivascular niche,3413,0.000924,0.000407
2,Tubular injury niche,217,0.000909,0.000349
3,Lymphoid infiltrate niche,144,0.000746,0.000319
4,Proximal straight nephron niche,3230,0.000437,0.000315
5,Collecting duct intercalated cell-driven niche,5181,0.000507,0.000199
6,Collecting duct principal cell-driven niche,2184,0.000451,0.000196
7,Proximal convoluted nephron niche,9849,0.000395,0.000178
8,Loop of Henle thick ascending limb-driven niche,6860,0.000316,0.000146
9,Collecting duct connecting tubule-driven niche,103,0.000239,0.000102
